(a) Consider a GPT-2 XL-sized model using our assignment architecture, which has the following configuration:

```yaml
vocab_size: 50,257
context_length: 1,024
num_layers: 48
d_model: 1,600
num_heads: 25
d_ff: 4,288 (the nearest multiple of 64 to 8/3 × 1, 600)
```

Suppose we constructed our model using this configuration.

1) How many trainable parameters would our model have? 

2) Assuming each parameter is represented using single-precision floating point, how much memory is required to just load this model?

**Deliverable**: A one-to-two sentence response.


In [1]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4288

In [5]:
# 1. Trainable parameters ()

# 1.1 Outside Transformer Block parameters
embed_params = vocab_size * d_model
rms_norm_after_transformer_block = d_model
linear_layer_final = d_model * vocab_size

# 1.2 Tranformer Block parameters
rms_norm_params_inside_transformer_block = d_model * 2

# 1.2.1 MutliHead Attention
q_proj = d_model * d_model
k_proj = d_model * d_model
v_proj = d_model * d_model
o_proj = d_model * d_model
multi_head_attention = q_proj + k_proj + v_proj + o_proj

# 1.2.2. SwiGLU
swiglu_matricies = d_ff * d_model * 3

# 1.3 General Result
trainable_parameters_tranformer_blocks = (
    (multi_head_attention + rms_norm_params_inside_transformer_block + swiglu_matricies) * num_layers
)
trainable_parameters = (
    embed_params + rms_norm_after_transformer_block + linear_layer_final + trainable_parameters_tranformer_blocks
)
print(f"1) {trainable_parameters} parameters (without weight tying)")

# 2. How much memory is required to just load this model?

print(f"2) {(trainable_parameters * 4 / 1024 / 1024 / 1042):.6f} Gb is required to train this model")


1) 1640452800 parameters (without weight tying)
2) 6.005596 Gb is required to train this model


(b) Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped model. How many FLOPs do these matrix multiplies require in total? Assume that our input sequence has context_length tokens.

**Deliverable**: A list of matrix multiplies (with descriptions), and the total number of FLOPs
required.

> Rule: Given 𝐴 ∈ ℝ𝑚×𝑛 and 𝐵 ∈ ℝ𝑛×𝑝, the matrix-matrix product 𝐴𝐵 requires 2𝑚𝑛𝑝 FLOPs.

In [ ]:
# 1) We get embedding matrix. It's not a matrix mutliplication, so we don't consider this operation 
# But after embeddings we have matrix with size of [context_length, d_model]

input_matrix = (context_length, d_model)
input_matrix_multiplied = input_matrix[0] * input_matrix[1]

# 2) Then we have rms_norm layer inside transformer block, but it's mosly element-wise multiplication

rms_layer_inside_transformer_block = d_model
"""
rms_a = torch.sqrt(1 / self.d_model * torch.sum(x * x, dim=-1, keepdim=True) + self.eps)
result = x * self.weight / rms_a
"""
rms_layer_transformer_block_flops = 2 * (input_matrix_multiplied * 4 + context_length * 3)

# 2.1 SwiGLU
silu_activation = 

13113344